## Đọc toàn bộ tập dữ liệu DUC_TEXT train

Mỗi khóa trong `txt_dict_by_file` là tên tệp; giá trị tương ứng là danh sách các câu được trích xuất từ thẻ `<s>`.

In [1]:
from pathlib import Path
from typing import Union
from bs4 import BeautifulSoup
from bs4.element import Tag

# Hỗ trợ chạy notebook từ thư mục dự án hoặc từ thư mục notebooks.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train_dir = project_root / "data" / "DUC_TEXT" / "train"

if not train_dir.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu: {train_dir}")

def get_string_attribute(tag: Tag, attribute_name: str) -> Union[str, None]:
    attribute_value = tag.get(attribute_name)
    return attribute_value if isinstance(attribute_value, str) else None

txt_dict_by_file: dict[str, list[dict[str, Union[str, None]]]] = {}

for file_path in sorted(train_dir.iterdir()):
    if not file_path.is_file():
        continue

    file_content = file_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(file_content, "html.parser")

    txt_dict_by_file[file_path.name] = [
        {
            "docid": get_string_attribute(tag, "docid"),
            "num": get_string_attribute(tag, "num"),
            "wdcount": get_string_attribute(tag, "wdcount"),
            "content": tag.get_text(separator="", strip=True),
        }
        for tag in soup.find_all("s")
    ]

total_paragraphs = sum(len(paragraphs) for paragraphs in txt_dict_by_file.values())
print(f"Đã đọc {len(txt_dict_by_file)} tệp với {total_paragraphs} đoạn văn")

# Xem thử dữ liệu của tệp đầu tiên.
first_file_name = next(iter(txt_dict_by_file), None)
if first_file_name is not None:
    print(f"Tệp đầu tiên: {first_file_name}")
    print(txt_dict_by_file[first_file_name][:3])

Đã đọc 50 tệp với 14707 đoạn văn
Tệp đầu tiên: d061j
[{'docid': 'AP880911-0016', 'num': '9', 'wdcount': '28', 'content': 'Hurricane Gilbert swept toward the Dominican Republic Sunday, and the Civil Defense alerted its heavily populated south coast to prepare for high winds, heavy rains and high seas.'}, {'docid': 'AP880911-0016', 'num': '10', 'wdcount': '17', 'content': 'The storm was approaching from the southeast with sustained winds of 75 mph gusting to 92 mph.'}, {'docid': 'AP880911-0016', 'num': '11', 'wdcount': '20', 'content': "`There is no need for alarm,'' Civil Defense Director Eugenio Cabral said in a television alert shortly before midnight Saturday."}]


## Đếm số lần xuất hiện của từ

Nội dung được chuyển thành chữ thường và loại bỏ dấu câu trước khi đếm.

In [2]:
import re
from collections import Counter

word_counts: Counter[str] = Counter()

for sentences in txt_dict_by_file.values():
    for sentence in sentences:
        content = sentence["content"] or ""
        words = re.findall(r"\b[\w']+\b", content.lower(), flags=re.UNICODE)
        word_counts.update(words)

total_words = sum(word_counts.values())

print(f"Tổng số từ: {total_words}")
print(f"Số từ khác nhau: {len(word_counts)}")
print("20 từ xuất hiện nhiều nhất:")

for word, count in word_counts.most_common(500):
    print(f"{word}: {count}")

Tổng số từ: 276547
Số từ khác nhau: 18995
20 từ xuất hiện nhiều nhất:
the: 18023
of: 7926
and: 6839
to: 6209
in: 6111
a: 5929
said: 3273
was: 2662
that: 2489
for: 2230
on: 1963
is: 1701
at: 1666
he: 1621
with: 1578
it: 1554
as: 1440
by: 1409
were: 1402
from: 1394
his: 1161
be: 1035
an: 1006
but: 987
had: 971
have: 943
who: 848
has: 816
i: 808
people: 806
they: 781
are: 781
not: 769
their: 703
about: 692
this: 675
one: 667
will: 636
her: 626
been: 617
she: 613
after: 574
all: 574
its: 568
would: 552
we: 546
when: 543
which: 537
more: 534
two: 526
or: 500
000: 492
there: 488
no: 477
than: 472
also: 471
up: 453
other: 442
out: 427
government: 406
_: 397
into: 395
year: 394
new: 389
most: 384
some: 381
years: 372
first: 368
soviet: 367
kuwait: 358
time: 352
could: 346
s: 340
1: 336
president: 333
u: 330
last: 316
if: 316
west: 306
east: 306
hurricane: 304
over: 301
state: 297
united: 294
miles: 291
only: 290
them: 288
iraq: 288
news: 286
city: 285
him: 278
officials: 273
many: 273
states: 